In [ ]:
# Importar las librerías necesarias
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
# Cargar los conjuntos de datos de entrenamiento y prueba
df_train = pd.read_csv('../data/fraudTrain.csv')
df_test = pd.read_csv('../data/fraudTest.csv')

In [ ]:
# Mostrar las dimensiones de los DataFrames
print("Train dataset shape:", df_train.shape)
print("Test dataset shape:", df_test.shape)

In [ ]:
# Mostrar los nombres de todas las columnas en el dataset de entrenamiento
print(df_train.columns)

In [ ]:
# Contar valores nulos por columna para identificar datos incompletos en entrenamiento
missing = df_train.isna().sum()

# Filtrar y mostrar solo las columnas que contienen al menos un valor nulo
print(missing[missing > 0])

In [ ]:
# Convertir la columna 'trans_date_trans_time' de texto a formato datetime de pandas
df_train['trans_date_trans_time'] = pd.to_datetime(df_train['trans_date_trans_time'])

# Verificar que el tipo de dato se haya actualizado correctamente
df_train['trans_date_trans_time'].dtype

In [ ]:
# Extraer la hora de la transacción y guardarla en una nueva columna    
df_train['trans_hour'] = df_train['trans_date_trans_time'].dt.hour

In [ ]:
# Calcular el número total de casos de fraude en entrenamiento y prueba
train_total_fraud_cases = df_train['is_fraud'].sum()
test_total_fraud_cases = df_test['is_fraud'].sum()

# Mostrar el total de casos de fraude detectados en cada dataset
print(f"Total fraud cases in the training dataset: {train_total_fraud_cases}")
print(f"Total fraud cases in the test dataset: {test_total_fraud_cases}")

# Calcular la tasa de fraude dividiendo los casos positivos por el total de registros
train_fraud_rate = train_total_fraud_cases / df_train.shape[0]
test_fraud_rate = test_total_fraud_cases / df_test.shape[0]

# Mostrar el porcentaje exacto de fraude para ambos datasets
print(f"Fraud rate in the training dataset: {train_fraud_rate: .2%}")
print(f"Fraud rate in the test dataset: {test_fraud_rate: .2%}")

In [ ]:
# Mostrar las primeras 5 filas del DataFrame para inspeccionar su estructura
df_train.head()

In [ ]:
# Dividir el DataFrame en dos subconjuntos: transacciones fraudulentas y legítimas
train_fraud_transactions = df_train[df_train['is_fraud'] == 1]
train_legitimate_transactions = df_train[df_train['is_fraud'] == 0]

# Generar estadísticas descriptivas
print("Summary statistics for fraudulent transactions:\n", train_fraud_transactions['amt'].describe())
print("Summary statistics for legitimate transactions:\n",train_legitimate_transactions['amt'].describe())

In [ ]:
# Calcular el monto promedio y mediana agrupado por categoría de compra y tipo de transacción 
df_train.groupby(['category', 'is_fraud'])['amt'].agg(['mean', 'median'])

In [ ]:
# Contar casos de fraude por categoría de comercio y ordenarlos de forma descendente
frauds_by_category = train_fraud_transactions['category'].value_counts().sort_values(ascending=False)

frauds_by_category

In [ ]:
# Contar transacciones legítimas por categoría para verificar el volumen en cada una
legitimate_transactions_by_category = train_legitimate_transactions['category'].value_counts().sort_values(ascending=False)
legitimate_transactions_by_category

In [ ]:
# Contar el número total de casos de fraude por género
frauds_by_gender = train_fraud_transactions['gender'].value_counts().sort_values(ascending=False)

# Generar una tabla cruzada que muestre la tasa exacta de fraude por género
pd.crosstab(df_train['gender'], df_train['is_fraud'], normalize='index') * 100

In [ ]:
# Agrupar y contar casos de fraude por hora del día
fraud_cases_by_hour = train_fraud_transactions['trans_hour'].value_counts().sort_index()
fraud_hours = fraud_cases_by_hour.index
fraud_case_counts = fraud_cases_by_hour.values
print(f"Fraud Cases by Hour of the Day:\n{fraud_cases_by_hour}")

In [ ]:
# Agrupar y contar transacciones legítimas por hora del día
legitimate_transactions_by_hour = train_legitimate_transactions['trans_hour'].value_counts().sort_index()
legitimate_hours = legitimate_transactions_by_hour.index
legitimate_transaction_counts = legitimate_transactions_by_hour.values
print(f"Legitimate Transactions by Hour of the Day:\n{legitimate_transactions_by_hour}")


In [ ]:
# Extraer el día de la semana para transacciones legítimas y contar su frecuencia
train_legitimate_transactions['trans_day'] = train_legitimate_transactions['trans_date_trans_time'].dt.day_name()
legitimate_transactions_by_day = train_legitimate_transactions['trans_day'].value_counts().sort_index()
print(f"Legitimate Transactions by Day:\n{legitimate_transactions_by_day}")

In [ ]:
# Extraer el día de la semana para transacciones fraudulentas y contar su frecuencia
train_fraud_transactions['trans_day'] = train_fraud_transactions['trans_date_trans_time'].dt.day_name()
fraud_cases_by_day = train_fraud_transactions['trans_day'].value_counts().sort_index()
print(f"Fraud Cases by Day:\n{fraud_cases_by_day}")

In [ ]:
# Calcular la tasa de fraude por día dividiendo casos de fraude sobre el total de transacciones
fraud_rate_by_day = fraud_cases_by_day / (fraud_cases_by_day + legitimate_transactions_by_day)

print(f'The fraud rate by day is: {fraud_rate_by_day.sort_values(ascending=False)}')

### Principales Hallazgos del EDA e Insights de Negocio

* **Concentración en Montos Altos:** Las transacciones fraudulentas se concentran fuertemente en montos superiores comparados con las compras legítimas, elevando considerablemente tanto la media como la mediana del valor de las operaciones.
* **Patrón Nocturno:** La gran mayoría de los eventos de fraude ocurren entre las 22:00 y las 03:00, estableciendo una ventana temporal clara de alto riesgo.
* **Disparidad por Género:** Los titulares de tarjeta de género masculino muestran una tasa de fraude del 0.64%, aproximadamente un 22% más alta que la de las mujeres (0.52%).
* **Variación Semanal**: La tasa de fraude fluctúa a lo largo de la semana sin seguir una regla binaria estricta de "días laborales vs. fin de semana".
* **Penetración por Categoría**: Ninguna categoría de comercio es inmune (todas registraron al menos una transacción fraudulenta), aunque los grupos con mayor volumen se concentran en compras presenciales de supermercado (`grocery_pos`) y compras en línea (`shopping_net`).

In [ ]:
# Calcular el volumen total de transacciones (legítimas + fraudulentas) por hora del día
total_transactions_by_hour = df_train['trans_hour'].value_counts().sort_index()

# Calcular la tasa relativa de fraude por hora para evitar el sesgo del volumen total de transacciones
fraud_rate_by_hour = (fraud_cases_by_hour / total_transactions_by_hour) * 100

# Crear una figura de Matplotlib
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Subgráfico izquierdo: Gráfico de barras con el número total de casos de fraude por hora
sns.barplot(ax=axes[0], x=fraud_cases_by_hour.index, y=fraud_cases_by_hour.values, color="#4c72b0")
axes[0].set_title("Total Fraud Cases by Hour")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Number of Cases")
axes[0].set_xlim(-1, 24)

# Subgráfico derecho: Gráfico de barras con la tasa de fraude (%) por hora
sns.barplot(ax=axes[1], x=fraud_rate_by_hour.index, y=fraud_rate_by_hour.values, color="#C76875")
axes[1].set_title("Fraud Rate (%) by Hour")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Fraud Rate (%)")
axes[1].set_xlim(-1, 24)

# Ajustar automáticamente el espacio entre subgráficos para evitar que se superpongan
plt.tight_layout()
plt.show()

In [ ]:
# Recalcular la cantidad de fraudes por categoría para asegurar el orden descendente en el gráfico
frauds_by_category = train_fraud_transactions['category'].value_counts()

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Crear un gráfico de barras horizontal
sns.barplot(
    x=frauds_by_category.values,     
    y=frauds_by_category.index,      
    color="#4c72b0"
)

# Configurar el título y las etiquetas de los ejes (tamaño de fuente y margen)
plt.title("Fraud Cases by Purchase Category", fontsize=14, pad=15)
plt.xlabel("Number of Cases", fontsize=12)
plt.ylabel("Category", fontsize=12)

plt.show()

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 6))

# Crear un diagrama de caja (boxplot) comparando montos de transacción ('amt') por tipo de transacción ('is_fraud')
sns.boxplot(
    data=df_train,
    x='is_fraud',
    y='amt',
    hue='is_fraud',
    palette=["#4c72b0", "#C76875"],
    fliersize=2
)

# Aplicar escala logarítmica al eje Y para visualizar mejor distribuciones con alto sesgo y valores atípicos
plt.yscale('log')

# Reemplazar los valores numéricos del eje X [0, 1] por etiquetas descriptivas
plt.xticks(ticks=[0, 1], labels=['Legitimate (0)', 'Fraud (1)'])

# Configurar títulos y etiquetas descriptivas con sus respectivos tamaños de fuente
plt.title("Transaction Amount Comparison: Legitimate vs. Fraudulent Transactions (Log Scale)", fontsize=13, pad=15)
plt.xlabel("Transaction Type", fontsize=11)
plt.ylabel("Transaction Amount (USD, Log Scale)", fontsize=11)

plt.show()